# extraccion -> fase 1

In [30]:
import pandas as pd

ventas = pd.read_json("ventas.json")


#FASE 1: -> 1
# ventas.info()

#FASE 1: -> 2
cantidad_nulos = ventas.isna().sum()
serie_2 = ventas.describe()
serie_3 = ventas.nunique()

#cantidad de datos al comienzo

print(f"cantidad de datos: {len(ventas)}")



cantidad de datos: 2003


analizando estos datos puedo llegar a que , el tipo de dato que recibe la columna fecha deberia seer del tipo date, el campo de precio tambien hay que cambiarlo al tipo numerico

# transformacion -> fase 2

### punto 3

In [31]:

ventas["ciudad"].unique()


<StringArray>
['Formosa', 'Resistencia', ' FORMOSA ', 'formosa', 'Clorinda']
Length: 5, dtype: str

In [32]:

#transformacion de ciudad
ventas["ciudad"] = ventas["ciudad"].str.strip().str.upper()


In [33]:

ventas["ciudad"].unique()


<StringArray>
['FORMOSA', 'RESISTENCIA', 'CLORINDA']
Length: 3, dtype: str

### Punto 4

In [34]:
ventas["precio"].unique()

array(['25000', 15000, 25000, 35000, 120000, 850000, -1000, -25000, -500,
       -15000, None, 0, '', 'gratis'], dtype=object)

In [35]:
# tranmsformacion de precio
precios_convertidos = pd.to_numeric(ventas["precio"], errors="coerce")
ventas_invalidas = ventas[precios_convertidos.isna()]
ventas["precio"] = precios_convertidos

# precios_convertidos
# ventas_invalidas
ventas["precio"]

0        25000.0
1        15000.0
2        25000.0
3        25000.0
4        35000.0
          ...   
1998     15000.0
1999     35000.0
2000    850000.0
2001    120000.0
2002     25000.0
Name: precio, Length: 2003, dtype: float64

In [36]:
ventas["precio"].unique()

array([ 2.5e+04,  1.5e+04,  3.5e+04,  1.2e+05,  8.5e+05, -1.0e+03,
       -2.5e+04, -5.0e+02, -1.5e+04,      nan,  0.0e+00])

### Punto 5

In [37]:

# analiso mas datos

ventas["cantidad"].min()



np.float64(-10.0)

In [38]:
ventas["cantidad"].max()



np.float64(5.0)

In [39]:
ventas["precio"].min()


np.float64(-25000.0)

In [40]:

ventas["precio"].max()

np.float64(850000.0)

In [41]:
# condiciono  y filtro

ventas_ok = ventas[
    (ventas["cantidad"] > 0) & (ventas["cantidad"] <= 5) & 
    (ventas["precio"] > 0) & (ventas["precio"] <= 50000)
]


ventas_bad = ventas[
    ~((ventas["cantidad"] > 0) & (ventas["cantidad"] <= 5) &
      (ventas["precio"] > 0) & (ventas["precio"] <= 5)
    )
]



In [42]:
ventas = ventas_ok

### Punto 6

In [43]:
ventas["estado"].unique()

<StringArray>
[  'pendiente',     'enviado',   'entregado',            '', 'desconocido',
           nan,  'pendiente ',   'cancelado',   'Entregado']
Length: 9, dtype: str

In [44]:
ventas["estado"] = ventas["estado"].str.strip().str.lower()

In [45]:
estados_validos = ["pendiente", "enviado", "entregado"]

estados_validos_ventas = ventas["estado"].isin(estados_validos)
estados_ivalidos_ventas = ventas[~ventas["estado"].isin(estados_validos)]

ventas = ventas[estados_validos_ventas]
ventas.head()

,id,producto,ciudad,precio,cantidad,estado,fecha,comentario_interno
0,1,Notebook,FORMOSA,25000.0,2.0,pendiente,2026-08-05,NaN
1,2,Notebook,RESISTENCIA,15000.0,5.0,enviado,2026-08-02,NaN
2,3,Notebook,FORMOSA,25000.0,2.0,entregado,2026-08-20,NaN
3,4,Notebook,RESISTENCIA,25000.0,5.0,enviado,2026-08-08,NaN
4,5,Monitor,RESISTENCIA,35000.0,1.0,pendiente,2026-08-23,NaN


### Punto 7

In [46]:
ventas["fecha"].dtype


<StringDtype(storage='python', na_value=nan)>

In [47]:
fechas_validas = pd.to_datetime(ventas["fecha"], format="%Y-%m-%d", errors="coerce")

fechas_invalidas = ventas[fechas_validas.isna()]

In [48]:
ventas["fecha"] = fechas_validas
ventas["fecha"].dtype

dtype('<M8[us]')

### Punto 8

Yo no considero que el campo de comentario interno no aporta nada 

In [49]:
ventas.columns

Index(['id', 'producto', 'ciudad', 'precio', 'cantidad', 'estado', 'fecha',
       'comentario_interno'],
      dtype='str')

In [50]:
ventas = ventas.drop(columns="comentario_interno")

In [51]:
ventas.columns

Index(['id', 'producto', 'ciudad', 'precio', 'cantidad', 'estado', 'fecha'], dtype='str')

### Punto 9

In [52]:
cant_duplicados = ventas.duplicated().sum()
cant_duplicados


np.int64(1)

In [53]:
ventas = ventas.drop_duplicates()


In [54]:
ventas.duplicated().sum()

np.int64(0)

# Fase 3 : carga

In [55]:
print(f"cantidad total de ventas procesadas: {len(ventas)}")

cantidad total de ventas procesadas: 1191


In [56]:
ventas_limpias = ventas[
    (ventas["cantidad"] > 0) &
    (ventas["cantidad"] <= 5) &
    (ventas["precio"] > 0) &
    (ventas["precio"] <= 50000) &
    (ventas["estado"].isin(estados_validos)) &
    (ventas["fecha"].notna())
]

print(f"cantidad de ventas validas: {len(ventas_limpias)}")
print(f"cantidad de duplicados ")


cantidad de ventas validas: 1187
cantidad de duplicados 


In [61]:
ventas_limpias.to_json("ventas_limpias.json", 
                       index=False,
                        orient="records",
                        force_ascii=False,
                        indent=4)

C:\Users\ivane\AppData\Local\Temp\ipykernel_12524\3156174180.py:1: Pandas4Warning: The default 'epoch' date format is deprecated and will be removed in a future version, please use 'iso' date format instead.
  ventas_limpias.to_json("ventas_limpias.json",


notas para mi 
ventas_limpias: DataFrame que se quiere guardar.

.to_json(): convierte el DataFrame a JSON.

"ventas_limpias.json": nombre del archivo.

orient="records": convierte cada fila en un objeto JSON.

force_ascii=False: conserva caracteres especiales.

indent=4: hace que el archivo quede ordenado y legible.